In [0]:
SELECT * FROM com_edp_prd.com_raw.vod_hcp LIMIT 5;

-- primary_specialty_group__v 
-- all_spec_cda__v
-- all_spec_group_cda__v
-- spec_1_cda__v
-- spec_group_1_cda__v

-- com_edp_prd.com_raw.vod_npi

-- com_edp_prd.com_raw.vod_primary_npi

-- com_edp_prd.reltio_in_out.vw_vod_hcp_hco_relationship

In [0]:
SELECT distinct primary_specialty_group__v, all_spec_cda__v, all_spec_group_cda__v, spec_1_cda__v, spec_group_1_cda__v , specialty_1__v
FROM com_edp_prd.com_raw.vod_hcp;


In [0]:
SELECT DISTINCT reference_type FROM com_edp_prd.com_raw.vod_references ;


In [0]:
    SELECT *
    FROM com_edp_prd.com_raw.vod_references
    WHERE code = 'CSPP';

In [0]:
    SELECT *
    FROM com_edp_prd.com_raw.vod_references
    WHERE reference_type IN 
    ('Specialty' , 'HCPSpecialtyCDA',
     'HCPSpecialtyGroupCDA', 'SpecialtyGroup')
      AND (
            name ILIKE '%Clinical Pharmacology%' OR
            name ILIKE '%Pharmacology%' OR
            name ILIKE '%Pharmacy Specialty%' OR
            name ILIKE '%Pharmaceutical Medicine%'
      );

In [0]:
SELECT DISTINCT(SPECIALTY_1__V) FROM com_edp_prd.com_raw.vod_hcp where primary_specialty_group__v  in ('G-PHM')

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent AS 

WITH specialty_ref AS (
    SELECT code
    FROM com_edp_prd.com_raw.vod_references
    WHERE reference_type = 'Specialty'
      AND (
            name ILIKE '%Clinical Pharmacology%' OR
            name ILIKE '%Pharmacology%' OR
            name ILIKE '%Pharmacy Specialty%' OR
            name ILIKE '%Pharmaceutical Medicine%'
      )
),

/* ================= HCP BASE ================= */
hcp_base AS (
    SELECT DISTINCT
        h.npi_num__v AS hcp_npi,
        h.first_name__v,
        h.last_name__v,
        h.specialty_1__v
    FROM com_edp_prd.com_raw.vod_hcp h
    INNER JOIN specialty_ref s
        ON h.specialty_1__v = s.code
    WHERE h.npi_num__v IS NOT NULL
),

/* ================= GET VID ================= */
hcp_vid AS (
    SELECT 
        a.*,
        b.vid__v AS hcp_vid
    FROM hcp_base a
    LEFT JOIN com_edp_prd.com_raw.vod_hcp b
        ON a.hcp_npi = b.npi_num__v
),

/* ================= VOD AFFILIATION ================= */
vod_ranked AS (
    SELECT
        a.hcp_npi,
        c.npi_num__v AS vod_hco_npi,
        c.corporate_name__v AS vod_hco_name,
        ROW_NUMBER() OVER (
            PARTITION BY a.hcp_npi
            ORDER BY 
                b.modified_date__v DESC NULLS LAST,
                b.status_update_time__v DESC NULLS LAST
        ) AS rn
    FROM hcp_vid a
    LEFT JOIN com_edp_prd.com_raw.vod_parenthco b
        ON a.hcp_vid = b.entity_vid__v
       AND b.hierarchy_type__v = 'HCP_HCO'
    LEFT JOIN com_edp_prd.com_raw.vod_hco c
        ON b.parent_hco_vid__v = c.vid__v
    WHERE b.parent_hco_status__v = 'A'
      AND b.relationship_type__v = '7356'
),

vod_final AS (
    SELECT
        hcp_npi,
        vod_hco_npi,
        vod_hco_name
    FROM vod_ranked
    WHERE rn = 1
),

/* ================= KOMODO FALLBACK ================= */
komodo AS (
    SELECT 
        a.hcp_npi,
        b.hco_primary_npi AS komodo_hco_npi,
        c.organization_name AS komodo_hco_name
    FROM hcp_base a
    LEFT JOIN com_edp_prd.com_raw.kom_providers b
        ON a.hcp_npi = b.npi
       AND b.provider_type = 'INDIVIDUAL'
    LEFT JOIN com_edp_prd.com_raw.kom_providers c
        ON b.hco_primary_npi = c.npi
       AND c.provider_type = 'ORGANIZATION'
),

/* ================= FINAL BASE ================= */
base_output AS (
    SELECT 
        a.hcp_npi,
        a.first_name__v,
        a.last_name__v,
        a.specialty_1__v AS hcp_specialty,

        COALESCE(v.vod_hco_npi, k.komodo_hco_npi, '-') AS reporting_hco_npi,
        COALESCE(v.vod_hco_name, k.komodo_hco_name, '-') AS reporting_hco_name,

        CASE 
            WHEN v.vod_hco_npi IS NOT NULL THEN 'VOD'
            WHEN k.komodo_hco_npi IS NOT NULL THEN 'Komodo'
            ELSE 'None'
        END AS affiliation_source

    FROM hcp_base a
    LEFT JOIN vod_final v ON a.hcp_npi = v.hcp_npi
    LEFT JOIN komodo k ON a.hcp_npi = k.hcp_npi
),

/* ================= VOD HCO ADDRESS ================= */
hco_vod_address AS (
    SELECT
        h.npi_num__v AS hco_npi,
        a.address_line_1__v,
        a.postal_code_cda__v,
        ROW_NUMBER() OVER (
            PARTITION BY h.npi_num__v
            ORDER BY a.modified_date__v DESC
        ) AS rn
    FROM com_edp_prd.com_raw.vod_hco h
    JOIN com_edp_prd.com_raw.vod_address a
        ON a.entity_vid__v = h.vid__v
       AND a.entity_type__v = 'HCO'
       AND a.record_state__v = 'VALID'
       AND a.address_status__v IN ('A','DS')
       AND a.address_verification_status__v NOT IN ('NS','U')
),

/* ================= FINAL ENRICHMENT ================= */
final_output AS (
    SELECT
        b.*,

        /* Address fallback logic */
        CASE 
            WHEN v.postal_code_cda__v IS NOT NULL THEN v.address_line_1__v
            ELSE kp.provider_address
        END AS reporting_parent_address,

        COALESCE(v.postal_code_cda__v, kp.provider_zip) AS reporting_parent_zip,
        z.city AS reporting_parent_city,
        z.state AS reporting_parent_state

    FROM base_output b

    LEFT JOIN hco_vod_address v
        ON b.reporting_hco_npi = v.hco_npi
       AND v.rn = 1

    LEFT JOIN com_edp_prd.com_raw.kom_providers kp
        ON b.reporting_hco_npi = kp.npi
       AND kp.provider_type = 'ORGANIZATION'

    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON COALESCE(v.postal_code_cda__v, kp.provider_zip) = z.zipcode
)

SELECT *
FROM final_output;

In [0]:
select count(distinct reporting_hco_npi) FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent

In [0]:
-- CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.pharmacist_hco_matched AS

/* ================= DISTINCT HCO ================= */
WITH hco_base AS (
    SELECT DISTINCT
        reporting_hco_name,
        reporting_parent_address,
        reporting_parent_zip,
        reporting_parent_city,
        reporting_parent_state
    FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent
    WHERE reporting_hco_name IS NOT NULL
      AND reporting_hco_name != '-'
),

/* ================= REF ================= */
ref AS (
    SELECT
        reporting_parent_name,
        reporting_parent_address,
        reporting_parent_zip,
        reporting_parent_city,
        reporting_parent_state,
        reporting_parent_tier,

        hco_name,
        hco_address,
        hco_zip,
        hco_city,
        hco_state,
        territory,
        region
    FROM com_edp_prd.cmpa_insights_internal_schema.reference_file_pooja_1703
    WHERE reporting_parent_tier IN ('Tier 1','Tier 2','Tier 3')
),

/* ================= FULL FUZZY ================= */
fuzzy_match AS (
    SELECT
        /* HCO */
        h.reporting_hco_name,
        h.reporting_parent_address AS hco_address,
        h.reporting_parent_zip     AS hco_zip,
        h.reporting_parent_city    AS hco_city,
        h.reporting_parent_state   AS hco_state,

        /* REF */
        r.reporting_parent_name,
        r.reporting_parent_address AS ref_address,
        r.reporting_parent_zip     AS ref_zip,
        r.reporting_parent_city,
        r.reporting_parent_state,
        r.reporting_parent_tier,

        r.hco_name,
        r.hco_address,
        r.hco_zip,
        r.hco_city,
        r.hco_state,
        r.territory,
        r.region,

        /* ================= SCORES ================= */

        1 - (
            levenshtein(lower(h.reporting_hco_name), lower(r.reporting_parent_name))
            / greatest(length(h.reporting_hco_name), length(r.reporting_parent_name),1)
        ) AS name_score,

        1 - (
            levenshtein(lower(h.reporting_parent_address), lower(r.reporting_parent_address))
            / greatest(length(h.reporting_parent_address), length(r.reporting_parent_address),1)
        ) AS address_score,

        CASE 
            WHEN h.reporting_parent_zip = r.reporting_parent_zip THEN 1 
            ELSE 0 
        END AS zip_score,

        /* FINAL SCORE */
        (
            (1 - (
                levenshtein(lower(h.reporting_hco_name), lower(r.reporting_parent_name))
                / greatest(length(h.reporting_hco_name), length(r.reporting_parent_name),1)
            )) * 0.5
          +
            (1 - (
                levenshtein(lower(h.reporting_parent_address), lower(r.reporting_parent_address))
                / greatest(length(h.reporting_parent_address), length(r.reporting_parent_address),1)
            )) * 0.3
          +
            (CASE WHEN h.reporting_parent_zip = r.reporting_parent_zip THEN 1 ELSE 0 END) * 0.2
        ) AS final_score

    FROM hco_base h
    CROSS JOIN ref r
),

/* ================= BEST MATCH ================= */
best_match AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY reporting_hco_name, hco_address
                   ORDER BY final_score DESC
               ) AS rn
        FROM fuzzy_match
        WHERE final_score > 0.7
    )
    WHERE rn = 1
)l

/* ================= FINAL ================= */
SELECT *
FROM best_match;

In [0]:
SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent;

-- SELECT COUNT(DISTINCT hcp_npi) FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent WHERE reporting_hco_npi ="-";

-- SELECT hcp_specialty, count(distinct hcp_npi) FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent group by 1 order by 2 desc;


-- SELECT COUNT(DISTINCT hcp_npi) FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent;

-- SELECT COUNT(DISTINCT hcp_npi) FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent where reporting_hco_name != "-" or reporting_hco_npi !="-";



In [0]:

SELECT *
FROM com_edp_prd.com_raw.vod_references
WHERE reference_type = 'Specialty'
AND (   
    name ILIKE '%Clinical Pharmacology%' OR
    name ILIKE '%Pharmacology%' OR
    name ILIKE '%Pharmacy Specialty%' OR
    name ILIKE '%Pharmaceutical Medicine%'
);

In [0]:
SELECT distinct name 
FROM com_edp_prd.com_raw.vod_references
WHERE reference_type = 'Specialty'
;

In [0]:
SELECT * FROM com_edp_prd.reltio_in_out.vw_vod_hcp_hco_relationship LIMIT 5;

In [0]:
-- SELECT * FROM com_edp_prd.com_raw.kom_providers LIMIT 5;

SELECT * FROM com_edp_prd.com_raw.kom_providers WHERE NPI ='1205896933';

In [0]:
WITH base_hcp AS (
    SELECT DISTINCT hcp_npi
    FROM your_hcp_list   -- replace with your table
),

/* ================= VOD ================= */

hcp_vid AS (
    SELECT 
        a.hcp_npi,
        b.vid__v AS hcp_vid
    FROM base_hcp a
    LEFT JOIN com_edp_prd.com_raw.vod_hcp b
        ON a.hcp_npi = b.npi_num__v
),

vod_ranked AS (
    SELECT
        a.hcp_npi,
        c.npi_num__v AS hco_npi,
        c.corporate_name__v AS hco_name,
        ROW_NUMBER() OVER (
            PARTITION BY a.hcp_npi
            ORDER BY 
                b.modified_date__v DESC NULLS LAST,
                b.status_update_time__v DESC NULLS LAST
        ) AS rn
    FROM hcp_vid a
    LEFT JOIN com_edp_prd.com_raw.vod_parenthco b
        ON a.hcp_vid = b.entity_vid__v
       AND b.hierarchy_type__v = 'HCP_HCO'
    LEFT JOIN com_edp_prd.com_raw.vod_hco c
        ON b.parent_hco_vid__v = c.vid__v
    WHERE b.parent_hco_status__v = 'A'
      AND b.relationship_type__v = '7356'
),

vod_final AS (
    SELECT 
        hcp_npi,
        hco_npi AS vod_hco_npi,
        hco_name AS vod_hco_name
    FROM vod_ranked
    WHERE rn = 1
),

/* ================= KOMODO ================= */

komodo AS (
    SELECT 
        a.hcp_npi,
        b.hco_primary_npi AS komodo_hco_npi,
        c.organization_name AS komodo_hco_name
    FROM base_hcp a
    LEFT JOIN com_edp_prd.com_raw.kom_providers b
        ON a.hcp_npi = b.npi
       AND b.provider_type = 'INDIVIDUAL'
    LEFT JOIN com_edp_prd.com_raw.kom_providers c
        ON b.hco_primary_npi = c.npi
       AND c.provider_type = 'ORGANIZATION'
),

/* ================= FINAL ================= */

final_affiliation AS (
    SELECT 
        a.hcp_npi,

        /* Final HCO (VOD priority) */
        COALESCE(v.vod_hco_npi, k.komodo_hco_npi, '-') AS final_hco_npi,
        COALESCE(v.vod_hco_name, k.komodo_hco_name, '-') AS final_hco_name,

        /* Debug columns */
        v.vod_hco_npi,
        v.vod_hco_name,
        k.komodo_hco_npi,
        k.komodo_hco_name,

        CASE 
            WHEN v.vod_hco_npi IS NOT NULL THEN 'VOD'
            WHEN k.komodo_hco_npi IS NOT NULL THEN 'Komodo'
            ELSE 'None'
        END AS affiliation_source

    FROM base_hcp a
    LEFT JOIN vod_final v ON a.hcp_npi = v.hcp_npi
    LEFT JOIN komodo k ON a.hcp_npi = k.hcp_npi
)

SELECT *
FROM final_affiliation;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_enriched AS

WITH ref AS (
    SELECT *
    FROM com_edp_prd.cmpa_insights_internal_schema.reference_file_pooja_1703
    WHERE reporting_parent_tier IN ('Tier 1','Tier 2','Tier 3')
),

pharma AS (
    SELECT *
    FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent where reporting_hco_name != "-" or reporting_hco_npi !="-"
),

/* ================= FUZZY MATCH ================= */

fuzzy_match AS (
    SELECT
        p.*,

        r.hcp_npi AS ref_hcp_npi,
        r.hcp_name AS ref_hcp_name,

        r.reporting_parent_name       AS ref_reporting_parent_name,
        r.reporting_parent_address    AS ref_reporting_parent_address,
        r.reporting_parent_zip        AS ref_reporting_parent_zip,
        r.reporting_parent_city       AS ref_reporting_parent_city,
        r.reporting_parent_state      AS ref_reporting_parent_state,
        r.reporting_parent_tier       AS ref_reporting_parent_tier,

        /* HCO details */
        r.hco_name,
        r.hco_address,
        r.hco_zip,
        r.hco_city,
        r.hco_state,
        r.territory,
        r.region,

        /* ================= Similarity scores (UPDATED) ================= */

        /* Name */
        (1 - levenshtein(lower(p.reporting_hco_name), lower(r.reporting_parent_name)) 
            / greatest(length(p.reporting_hco_name), length(r.reporting_parent_name),1)
        ) AS name_score,

        /* ✅ Address comparison now uses PHARMA table */
        (1 - levenshtein(lower(p.reporting_parent_address), lower(r.reporting_parent_address)) 
            / greatest(length(p.reporting_parent_address), length(r.reporting_parent_address),1)
        ) AS address_score,

        /* ZIP match */
        CASE 
            WHEN p.reporting_parent_zip = r.reporting_parent_zip THEN 1 
            ELSE 0 
        END AS zip_score,

        /* ================= Combined score ================= */

        (
            (1 - levenshtein(lower(p.reporting_hco_name), lower(r.reporting_parent_name)) 
                / greatest(length(p.reporting_hco_name), length(r.reporting_parent_name),1)) * 0.4
          +
            (1 - levenshtein(lower(p.reporting_parent_address), lower(r.reporting_parent_address)) 
                / greatest(length(p.reporting_parent_address), length(r.reporting_parent_address),1)) * 0.3
          +
            (CASE WHEN p.reporting_parent_zip = r.reporting_parent_zip THEN 1 ELSE 0 END) * 0.3
        ) AS final_score

    FROM pharma p
    CROSS JOIN ref r
),

/* ================= FILTER BEST MATCH ================= */

best_match AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY hcp_npi ORDER BY final_score DESC) AS rn
        FROM fuzzy_match
        WHERE final_score > 0.7
    )
    WHERE rn = 1
),

/* ================= HCP ENRICHMENT ================= */

hcp_enriched AS (
    SELECT
        b.hcp_npi,

        CONCAT(v.first_name__v, ' ', v.last_name__v) AS hcp_name,
        v.first_name__v AS hcp_first_name,
        v.last_name__v AS hcp_last_name,

        v.postal_code_cda__v AS hcp_zip,
        z.state AS hcp_state,

        -- v.primary_email__v AS hcp_primary_email,

        COALESCE(kp.primary_specialty, v.specialty_1__v, '-') AS hcp_primary_specialty,
        COALESCE(kp.secondary_specialty, v.specialty_2__v, '-') AS hcp_secondary_specialty,

        -- v.veeva_network_id__v AS veeva_network_id,
        -- v.id AS veeva_crm_id,

        b.ref_reporting_parent_name     AS reporting_parent_name,
        b.ref_reporting_parent_address  AS reporting_parent_address,
        b.ref_reporting_parent_zip      AS reporting_parent_zip,
        b.ref_reporting_parent_city     AS reporting_parent_city,
        b.ref_reporting_parent_state    AS reporting_parent_state,
        b.ref_reporting_parent_tier     AS reporting_parent_tier,

        /* Duplicate fields */
        -- v.veeva_network_id__v AS veeva_network_id_2,
        -- v.id AS veeva_crm_id_2,

        /* HCO details */
        b.hco_name,
        b.hco_address,
        b.hco_zip,
        b.hco_city,
        b.hco_state,
        b.territory,
        b.region,

        /* ================= FLAGS ================= */

        CASE 
            WHEN r.hcp_npi IS NOT NULL THEN 1
            WHEN r.hcp_name IS NOT NULL 
                 AND r.hcp_zip = v.postal_code_cda__v THEN 1
            ELSE 0
        END AS is_present_in_reference_file,

        /* Debug */
        b.name_score,
        -- b.hco_name_score,
        b.address_score,
        b.zip_score,
        b.final_score

    FROM best_match b

    LEFT JOIN com_edp_prd.com_raw.vod_hcp v
        ON b.hcp_npi = v.npi_num__v

    LEFT JOIN com_edp_prd.com_raw.kom_providers kp
        ON b.hcp_npi = kp.npi
       AND kp.provider_type = 'INDIVIDUAL'

    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON v.postal_code_cda__v = z.zipcode

    LEFT JOIN ref r
        ON b.hcp_npi = r.hcp_npi
)

SELECT *
FROM hcp_enriched;


SELECT * FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_enriched ;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_enriched AS

/* ================= REF ================= */
WITH ref AS (
    SELECT 
        hcp_npi,
        hcp_name,
        reporting_parent_name,
        reporting_parent_address,
        reporting_parent_zip,
        reporting_parent_city,
        reporting_parent_state,
        reporting_parent_tier,
        hco_name,
        hco_address,
        hco_zip,
        hco_city,
        hco_state,
        territory,
        region,

        /* Precompute lowercase */
        LOWER(reporting_parent_name) AS ref_name_l,
        LOWER(reporting_parent_address) AS ref_addr_l,
        LOWER(reporting_parent_city) AS ref_city_l

    FROM com_edp_prd.cmpa_insights_internal_schema.reference_file_pooja_1703
    WHERE reporting_parent_tier IN ('Tier 1','Tier 2','Tier 3')
),

/* ================= PHARMA ================= */
pharma AS (
    SELECT 
        hcp_npi,
        reporting_hco_name,
        reporting_parent_address,
        reporting_parent_zip,
        reporting_parent_city,
        reporting_parent_state,

        /* Precompute lowercase */
        LOWER(reporting_hco_name) AS p_name_l,
        LOWER(reporting_parent_address) AS p_addr_l,
        LOWER(reporting_parent_city) AS p_city_l

    FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent
    WHERE reporting_hco_name != '-' OR reporting_hco_npi != '-'
),

/* ================= BLOCKING (CITY + ZIP/NAME) ================= */
candidate_pairs AS (
    SELECT
        p.*,
        r.*

    FROM pharma p
    JOIN ref r
        ON p.p_city_l = r.ref_city_l
       AND (
            p.reporting_parent_zip = r.reporting_parent_zip
            OR LEFT(p.p_name_l, 4) = LEFT(r.ref_name_l, 4)
       )
),

/* ================= DISTANCE CALCULATION ================= */
scored AS (
    SELECT
        *,

        levenshtein(p_name_l, ref_name_l) AS name_dist,
        levenshtein(p_addr_l, ref_addr_l) AS addr_dist,

        GREATEST(LENGTH(p_name_l), LENGTH(ref_name_l), 1) AS name_len,
        GREATEST(LENGTH(p_addr_l), LENGTH(ref_addr_l), 1) AS addr_len

    FROM candidate_pairs
),

/* ================= SCORING ================= */
fuzzy_match AS (
    SELECT
        *,

        1 - (name_dist / name_len) AS name_score,
        1 - (addr_dist / addr_len) AS address_score,

        CASE 
            WHEN reporting_parent_zip = reporting_parent_zip THEN 1 
            ELSE 0 
        END AS zip_score,

        (
            (1 - (name_dist / name_len)) * 0.4 +
            (1 - (addr_dist / addr_len)) * 0.3 +
            (CASE WHEN reporting_parent_zip = reporting_parent_zip THEN 1 ELSE 0 END) * 0.3
        ) AS final_score

    FROM scored
),

/* ================= BEST MATCH ================= */
best_match AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY hcp_npi ORDER BY final_score DESC) AS rn
        FROM fuzzy_match
        WHERE final_score > 0.7
    )
    WHERE rn = 1
),

/* ================= ENRICHMENT ================= */
hcp_enriched AS (
    SELECT
        b.hcp_npi,

        CONCAT(v.first_name__v, ' ', v.last_name__v) AS hcp_name,
        v.first_name__v AS hcp_first_name,
        v.last_name__v AS hcp_last_name,

        v.postal_code_cda__v AS hcp_zip,
        z.state AS hcp_state,

        COALESCE(kp.primary_specialty, v.specialty_1__v, '-') AS hcp_primary_specialty,
        COALESCE(kp.secondary_specialty, v.specialty_2__v, '-') AS hcp_secondary_specialty,

        /* Reporting parent */
        b.reporting_parent_name,
        b.reporting_parent_address,
        b.reporting_parent_zip,
        b.reporting_parent_city,
        b.reporting_parent_state,
        b.reporting_parent_tier,

        /* HCO */
        b.hco_name,
        b.hco_address,
        b.hco_zip,
        b.hco_city,
        b.hco_state,
        b.territory,
        b.region,

        /* Presence flag */
        CASE 
            WHEN r.hcp_npi IS NOT NULL THEN 1
            WHEN r.hcp_name IS NOT NULL 
                 AND r.reporting_parent_zip = v.postal_code_cda__v THEN 1
            ELSE 0
        END AS is_present_in_reference_file,

        /* Debug */
        b.name_score,
        b.address_score,
        b.final_score

    FROM best_match b

    LEFT JOIN com_edp_prd.com_raw.vod_hcp v
        ON b.hcp_npi = v.npi_num__v

    LEFT JOIN com_edp_prd.com_raw.kom_providers kp
        ON b.hcp_npi = kp.npi
       AND kp.provider_type = 'INDIVIDUAL'

    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON v.postal_code_cda__v = z.zipcode

    LEFT JOIN ref r
        ON b.hcp_npi = r.hcp_npi
)

/* ================= FINAL ================= */
SELECT *
FROM hcp_enriched;

In [0]:
CREATE OR REPLACE TABLE com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_enriched AS

/* ================= SMALLER REF ================= */
WITH ref AS (
    SELECT DISTINCT
        reporting_parent_name,
        reporting_parent_address,
        reporting_parent_zip,
        reporting_parent_city,
        reporting_parent_state,
        reporting_parent_tier,
        hco_name,
        hco_address,
        hco_zip,
        hco_city,
        hco_state,
        territory,
        region,
        hcp_npi,
        hcp_name
    FROM com_edp_prd.cmpa_insights_internal_schema.reference_file_pooja_1703
    WHERE reporting_parent_tier IN ('Tier 1','Tier 2','Tier 3')
),

/* ================= FILTER PHARMA ================= */
pharma AS (
    SELECT *
    FROM com_edp_prd.cmpa_insights_internal_schema.pharmacist_hcp_reporting_parent
    WHERE reporting_hco_name != '-' 
       OR reporting_hco_npi != '-'
),

/* ================= BLOCKING (CRITICAL) ================= */
candidate_match AS (
    SELECT
        p.*,

        r.hcp_npi AS ref_hcp_npi,
        r.hcp_name AS ref_hcp_name,

        r.reporting_parent_name       AS ref_reporting_parent_name,
        r.reporting_parent_address    AS ref_reporting_parent_address,
        r.reporting_parent_zip        AS ref_reporting_parent_zip,
        r.reporting_parent_city       AS ref_reporting_parent_city,
        r.reporting_parent_state      AS ref_reporting_parent_state,
        r.reporting_parent_tier       AS ref_reporting_parent_tier,

        r.hco_name,
        r.hco_address,
        r.hco_zip,
        r.hco_city,
        r.hco_state,
        r.territory,
        r.region

    FROM pharma p
    INNER JOIN ref r
        ON (
            /* 🔥 Primary: ZIP match */
            p.reporting_parent_zip = r.reporting_parent_zip
            
            /* 🔥 Fallback: Name prefix */
            OR substr(lower(p.reporting_hco_name),1,6) = substr(lower(r.reporting_parent_name),1,6)
        )
),

/* ================= FUZZY (ONLY ON CANDIDATES) ================= */
fuzzy_match AS (
    SELECT
        *,

        /* NAME */
        (1 - levenshtein(lower(reporting_hco_name), lower(ref_reporting_parent_name)) 
            / greatest(length(reporting_hco_name), length(ref_reporting_parent_name),1)
        ) AS name_score,

        /* ADDRESS */
        (1 - levenshtein(lower(reporting_parent_address), lower(ref_reporting_parent_address)) 
            / greatest(length(reporting_parent_address), length(ref_reporting_parent_address),1)
        ) AS address_score,

        /* ZIP */
        CASE 
            WHEN reporting_parent_zip = ref_reporting_parent_zip THEN 1 
            ELSE 0 
        END AS zip_score,

        /* FINAL */
        (
            (1 - levenshtein(lower(reporting_hco_name), lower(ref_reporting_parent_name)) 
                / greatest(length(reporting_hco_name), length(ref_reporting_parent_name),1)) * 0.5
          +
            (1 - levenshtein(lower(reporting_parent_address), lower(ref_reporting_parent_address)) 
                / greatest(length(reporting_parent_address), length(ref_reporting_parent_address),1)) * 0.3
          +
            (CASE WHEN reporting_parent_zip = ref_reporting_parent_zip THEN 1 ELSE 0 END) * 0.2
        ) AS final_score
),

/* ================= BEST MATCH ================= */
best_match AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY hcp_npi ORDER BY final_score DESC) AS rn
        FROM fuzzy_match
        WHERE final_score > 0.7
    )
    WHERE rn = 1
),

/* ================= FINAL ENRICHMENT ================= */
hcp_enriched AS (
    SELECT
        b.hcp_npi,

        CONCAT(v.first_name__v, ' ', v.last_name__v) AS hcp_name,
        v.first_name__v AS hcp_first_name,
        v.last_name__v AS hcp_last_name,

        v.postal_code_cda__v AS hcp_zip,
        z.state AS hcp_state,

        COALESCE(kp.primary_specialty, v.specialty_1__v, '-') AS hcp_primary_specialty,
        COALESCE(kp.secondary_specialty, v.specialty_2__v, '-') AS hcp_secondary_specialty,

        /* Reporting parent */
        b.ref_reporting_parent_name     AS reporting_parent_name,
        b.ref_reporting_parent_address  AS reporting_parent_address,
        b.ref_reporting_parent_zip      AS reporting_parent_zip,
        b.ref_reporting_parent_city     AS reporting_parent_city,
        b.ref_reporting_parent_state    AS reporting_parent_state,
        b.ref_reporting_parent_tier     AS reporting_parent_tier,

        /* HCO */
        b.hco_name,
        b.hco_address,
        b.hco_zip,
        b.hco_city,
        b.hco_state,
        b.territory,
        b.region,

        /* FLAG */
        CASE 
            WHEN r.hcp_npi IS NOT NULL THEN 1
            WHEN r.hcp_name IS NOT NULL 
                 AND r.hcp_zip = v.postal_code_cda__v THEN 1
            ELSE 0
        END AS is_present_in_reference_file,

        /* DEBUG */
        b.name_score,
        b.address_score,
        b.zip_score,
        b.final_score

    FROM best_match b

    LEFT JOIN com_edp_prd.com_raw.vod_hcp v
        ON b.hcp_npi = v.npi_num__v

    LEFT JOIN com_edp_prd.com_raw.kom_providers kp
        ON b.hcp_npi = kp.npi
       AND kp.provider_type = 'INDIVIDUAL'

    LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping z
        ON v.postal_code_cda__v = z.zipcode

    LEFT JOIN ref r
        ON b.hcp_npi = r.hcp_npi
)

SELECT *
FROM hcp_enriched;